# 5.15 一次比較略好就有進步嗎？

# 第 5 章：訓練與評估形成一個流程

前置：第4章的 logits／shift／loss。程式預設只做結構、梯度與記錄檢查。凡是涉及學到能力的對照，先給可執行配方，之後由讀者在 GPU 上跑；不會把隨機模型的輸出寫成訓練成果。




In [ ]:
from pathlib import Path
import sys

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("先依 course/README.md 下載並開啟這個 repo")
sys.path.insert(0, str(root))
import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)
torch.manual_seed(42)

**先看圖：一次更新的生命週期**

忘記清零會累加；累積梯度時才刻意保留。

橘色邊框只提示閱讀順序，靜態標註一直保留。系統的減少動態效果設定會停用動畫。


In [ ]:
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/backward.svg")))

**先想一想：**每種方法只挑最好seed，會發生什麼？

初始化與抽樣造成波動。比較方法時使用同一組seeds，報告每次值與範圍；三次只提供很粗的局部證據。

**把直覺寫成數學：**比較配對差值比只看兩組最高分更可靠；小樣本不能支撐精確置信結論。

**最小實驗：**以下程式只處理這節的問題。


In [ ]:
from tiny_perceptron.model import TinyLM, ModelConfig

for seed in (1, 2, 3):
    torch.manual_seed(seed)
    model = TinyLM(ModelConfig(width=8))
    print(seed, model.embedding.weight.std().item(), "品質待訓練後量測")

**如何讀結果：**這裏只顯示初始化變化；不同隨機權重的loss差不代表方法改善。

**只改一件事：**只固定相同seed重建兩次，確認初值一致。
